In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("data/maze.csv")

In [6]:
# Extract row and column indices from the 'cell' column
df[['row', 'col']] = (
df['  cell  '].astype(str).str.replace(r'[()]', '', regex=True).str.split(',', expand=True).astype(
        int))

In [7]:
df.head()

,cell,E,W,N,S,row,col
0,"(1, 1)",1,0,0,1,1,1
1,"(2, 1)",0,0,1,1,2,1
2,"(3, 1)",1,0,1,0,3,1
3,"(4, 1)",1,0,0,0,4,1
4,"(5, 1)",1,0,0,1,5,1


In [4]:
import csv

def generate_maze_csv(filename, rows=20, cols=20):
    """
    Generates a CSV file representing a 20x20 maze with 0 indicating no wall and 1 indicating a wall.
    Boundary cells have walls on the respective edges.
    """
    with open(filename, mode="w", newline="") as file:
        writer = csv.writer(file)
        
        # Write the header row
        writer.writerow(["  cell  ", "E", "W", "N", "S"])
        
        # Loop over each cell in the grid
        for y in range(1, rows + 1):
            for x in range(1, cols + 1):
                # Determine boundary walls (1 for wall, 0 for no wall)
                E = 0 if x == cols else 1  # East boundary if at the last column
                W = 0 if x == 1 else 1     # West boundary if at the first column
                N = 0 if y == 1 else 1     # North boundary if at the first row
                S = 0 if y == rows else 1  # South boundary if at the last row
                
                # Each cell is labeled as (x, y)
                writer.writerow([f"({x},{y})", E, W, N, S])

# Generate the 20x20 maze CSV
generate_maze_csv("data/test_maze.csv")


In [4]:
from pyamaze import maze,agent
m=maze(20,20)
m.CreateMaze(loopPercent=50)
a=agent(m,filled=True,footprints=True)
m.tracePath({a:m.path})
m.run()

TclError: bad argument "zoomed": must be normal, iconic, or withdrawn

In [2]:
from pyamaze import maze
import pandas as pd

def generate_df_maze(rows=20, cols=20):
    # Create a Pyamaze maze
    m = maze(rows, cols)
    m.CreateMaze(loopPercent=80)  # No loops to match a grid-like maze

    # List to store cell data
    maze_data = []

    # Loop through each cell in the maze
    for y in range(1, rows + 1):
        for x in range(1, cols + 1):
            cell = (x, y)
            cell_walls = m.maze_map[cell]  # Get wall info for each cell

            # Convert to E, W, N, S format (1 for wall, 0 for no wall)
            E = 0 if cell_walls['E'] else 1
            W = 0 if cell_walls['W'] else 1
            N = 0 if cell_walls['N'] else 1
            S = 0 if cell_walls['S'] else 1

            # Store cell information
            maze_data.append([f"({x},{y})", E, W, N, S])

    # Create DataFrame
    df = pd.DataFrame(maze_data, columns=["cell", "E", "W", "N", "S"])
    return df

# Generate the maze DataFrame
df_maze = generate_df_maze(20, 20)
print(df_maze.head())




TclError: bad argument "zoomed": must be normal, iconic, or withdrawn

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import ast

def draw_maze_and_tasks(maze_csv, num_tasks, num_patrols):
    # Read the CSV file
    if type(maze_csv) == "str":
        df = pd.read_csv(maze_csv)
    else:
        df = maze_csv
    
    # Parse the cell string into a tuple and extract row, col
    df['  cell  '] = df['  cell  '].apply(lambda x: ast.literal_eval(x))
    df['row'] = df['  cell  '].apply(lambda t: t[0])
    df['col'] = df['  cell  '].apply(lambda t: t[1])
    
    # Create a plot for the maze
    fig, ax = plt.subplots(figsize=(8, 8))
    
    # For each cell, draw walls where flagged (assume each cell spans from (col-1, row-1) to (col, row))
    for _, cell in df.iterrows():
        r, c = cell['row'], cell['col']
        # South wall: from (c-1, r-1) to (c, r-1)
        if cell['S'] == 0:
            ax.plot([c-1, c], [r-1, r-1], color='black')
        # North wall: from (c-1, r) to (c, r)
        if cell['N'] == 0:
            ax.plot([c-1, c], [r, r], color='black')
        # West wall: from (c-1, r-1) to (c-1, r)
        if cell['W'] == 0:
            ax.plot([c-1, c-1], [r-1, r], color='black')
        # East wall: from (c, r-1) to (c, r)
        if cell['E'] == 0:
            ax.plot([c, c], [r-1, r], color='black')
    
    # Set aspect ratio so cells appear square
    ax.set_aspect('equal')
    
    # Sample a set of tasks from the maze cells and extract (row, col)
    tasks_df = df[['row', 'col']].sample(n=num_tasks, random_state=47)
    task_list = [ (int(t[0]), int(t[1])) for t in tasks_df.values.tolist() ]
    
    # Cluster the tasks with KMeans
    kmeans = KMeans(n_clusters=num_patrols, random_state=47).fit(task_list)
    labels = kmeans.labels_
    
    # Plot the tasks on top of the maze.
    # We'll mark the center of each cell at (col - 0.5, row - 0.5)
    xs = [t[1] - 0.5 for t in task_list]  # x comes from col
    ys = [t[0] - 0.5 for t in task_list]  # y comes from row
    
    scatter = ax.scatter(xs, ys, c=labels, cmap='tab10', s=100, edgecolors='k')
    
    # Optionally, annotate each task with its cell coordinates.
    for i, (x, y) in enumerate(zip(xs, ys)):
        ax.text(x, y, str(task_list[i]), color='white', ha='center', va='center')
    
    plt.title("Maze with Task Clusters")
    plt.xlabel("Column")
    plt.ylabel("Row")
    
    # Optionally, invert y-axis if you want (e.g. so that (1,1) appears in the top-left)
    plt.gca().invert_yaxis()
    plt.show()

# Example usage:
# Assume your maze CSV is named 'maze.csv'
draw_maze_and_tasks(df_maze, num_tasks=40, num_patrols=7)


NameError: name 'df_maze' is not defined